# Step 1: Tokenization with BPE

**Goal:** turn text into a list of IDs, and back. The model never sees letters, only numbers.

BPE (byte pair encoding) **learns** its pieces from the training text. It repeats 3 moves:
1. **Count** every pair of side-by-side pieces inside each word. A word seen 3 times counts 3 times.
2. **Pick** the pair with the biggest count. On a tie, take the pair met first when reading the text.
3. **Glue** that pair into one new piece everywhere. Add it to the vocabulary and write it in the **merge list**.

How to read the printouts: `_` means "a word starts here". A space separates pieces: `_ w a n t` is 5 pieces, `_want` is 1 piece.

We build everything here, cell by cell. The same code lives in `toddler/bpe.py`, so later steps can import it.

In [ ]:
# Make the `toddler` package (repo root) importable from this notebook
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "toddler").is_dir())
sys.path.insert(0, str(ROOT))

from toddler.data import CORPUS
from toddler.exercises import check, solution

## Part A: Prepare the text

**A1.** Split into words and put `_` in front of every word. Later, it tells decoding where the spaces go.

In [ ]:
def to_words(text):
    return ["_" + word for word in text.split()]

for sentence in CORPUS:
    print(f"{sentence:<30} → {' '.join(to_words(sentence))}")

**A2.** Count the words.

In [ ]:
def count_words(corpus):
    counts = {}  # insertion order = order of first appearance in the text
    for text in corpus:
        for word in to_words(text):
            counts[word] = counts.get(word, 0) + 1
    return counts

word_counts = count_words(CORPUS)
for word, count in word_counts.items():
    print(f"{word:<8} ×{count}")
print("total:", sum(word_counts.values()), "words")

**A3.** Cut every word into single symbols. **A4.** The starting vocabulary is every different symbol.

In [ ]:
splits = {word: list(word) for word in word_counts}
for word, pieces in splits.items():
    print(f"{word:<8} → {' '.join(pieces)}")

start_vocab = []
for pieces in splits.values():
    for symbol in pieces:
        if symbol not in start_vocab:
            start_vocab.append(symbol)
print(f"\nstarting vocabulary, {len(start_vocab)} symbols: {' '.join(start_vocab)}")

## Part B: Training, round 1 in full detail

**Move 1: count** every side-by-side pair inside each word.

In [ ]:
def count_pairs(splits, word_counts):
    pairs = {}
    for word, pieces in splits.items():
        for pair in zip(pieces, pieces[1:]):
            pairs[pair] = pairs.get(pair, 0) + word_counts[word]
    return pairs

pairs = count_pairs(splits, word_counts)
for (left, right), count in sorted(pairs.items(), key=lambda item: -item[1]):  # biggest first; ties keep text order
    print(f"'{left} {right}'  {count}")

**Move 2: pick** the biggest count. On a tie, `max()` returns the first pair it meets, and `pairs` is in text order, so this applies our tie rule.

In [ ]:
best = max(pairs, key=pairs.get)
print("winner:", best, "with count", pairs[best])

**Move 3: glue** that pair everywhere it appears.

In [ ]:
def merge_pair(pair, pieces):
    left, right = pair
    out, i = [], 0
    while i < len(pieces):
        if i + 1 < len(pieces) and (pieces[i], pieces[i + 1]) == pair:
            out.append(left + right)
            i += 2
        else:
            out.append(pieces[i])
            i += 1
    return out

for word, pieces in splits.items():
    merged = merge_pair(best, pieces)
    if merged != pieces:
        print(f"{' '.join(pieces)}  →  {' '.join(merged)}")

## Part B: all the rounds

The same 3 moves in a loop, counting again after every glue. Each line is one round: the top pairs (with their count), the winner, and what changes. Compare it with the tables from the chat.

In [ ]:
def train_bpe(corpus, max_merges=None, verbose=False):
    word_counts = count_words(corpus)
    splits = {word: list(word) for word in word_counts}
    vocab = []
    for pieces in splits.values():
        for symbol in pieces:
            if symbol not in vocab:
                vocab.append(symbol)
    merges = []
    while max_merges is None or len(merges) < max_merges:
        pairs = count_pairs(splits, word_counts)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        new_splits = {word: merge_pair(best, pieces) for word, pieces in splits.items()}
        if verbose:
            top = [f"{a} {b}" for (a, b), count in pairs.items() if count == pairs[best]]
            top = ", ".join(top[:4]) + (", ..." if len(top) > 4 else "")
            winner = f"{best[0]} {best[1]}"
            changes = ";  ".join(f"{' '.join(splits[w])} → {' '.join(new_splits[w])}"
                                 for w in splits if new_splits[w] != splits[w])
            print(f"#{len(merges) + 1:<3} top ({pairs[best]}): {top:<28} winner: {winner:<12} | {changes}")
        splits = new_splits
        merges.append(best)
        vocab.append(best[0] + best[1])
    return merges, vocab

merges, vocab = train_bpe(CORPUS, verbose=True)

Training gives **two things**: the **vocabulary** (a token's position is its ID) and the **merge list** (the recipe for cutting new text).

In [ ]:
token_to_id = {token: i for i, token in enumerate(vocab)}
print(f"{len(vocab)} tokens = {len(start_vocab)} symbols + {len(merges)} merges\n")
for token_id, token in enumerate(vocab):
    print(f"{token_id:>2} {token:<8}", end="\n" if token_id % 8 == 7 else "")
print()

## Part C: Encoding

Put `_` before each word and cut it into symbols. Then go down the merge list **in order** (#1, #2, … #32): wherever a merge's pair appears, glue it. Finally, look up the IDs.

Here is every merge tried on the new word "wanted":

In [ ]:
def encode_word(word, merges, verbose=False, show_misses=False):
    pieces = list(word)
    if verbose:
        print(f"{'start':<16}{' '.join(pieces)}")
    for number, pair in enumerate(merges, start=1):
        merged = merge_pair(pair, pieces)
        found = merged != pieces
        if verbose and (found or show_misses):
            label = f"#{number} {pair[0]} + {pair[1]}"
            print(f"{label:<16}" + ("yes → " + " ".join(merged) if found else "no"))
        pieces = merged
    return pieces

pieces = encode_word("_wanted", merges, verbose=True, show_misses=True)
print("\nresult:", pieces, "→ IDs", [token_to_id[p] for p in pieces])

The same, showing only the merges that fire:

In [ ]:
for word in ["_milky", "_mom"]:
    print(f"--- {word}")
    pieces = encode_word(word, merges, verbose=True)
    print("result:", pieces, "→ IDs", [token_to_id[p] for p in pieces], "\n")

A word with a letter the toddler never heard:

In [ ]:
pieces = encode_word("_sleepy", merges, verbose=True)
print("\nnot in the vocabulary:", [p for p in pieces if p not in token_to_id])

`p` is not one of our 21 symbols, so "sleepy" can't be encoded. Real tokenizers start from the **256 possible bytes**: every character of every language is made of bytes, so nothing is ever unknown.

## Part D: Encode and decode whole sentences

In [ ]:
def encode(text):
    return [token_to_id[piece] for word in to_words(text) for piece in encode_word(word, merges)]

def decode(ids):
    text = "".join(vocab[i] for i in ids).replace("_", " ")
    return text[1:] if text.startswith(" ") else text

for sentence in CORPUS:
    ids = encode(sentence)
    print(f"{sentence:<30} → {ids} → {decode(ids)!r}")

Decoding, step by step:

In [ ]:
ids = [21, 28, 39, 9, 22]
pieces = [vocab[i] for i in ids]
glued = "".join(pieces)
print("IDs            ", ids)
print("pieces         ", pieces)
print("glued          ", repr(glued))
print("_ → space      ", repr(glued.replace("_", " ")))
print("drop 1st space ", repr(decode(ids)))

## The package version

The same code lives in `toddler/bpe.py` as `BPETokenizer`. Later steps use it like this:

In [ ]:
from toddler.bpe import BPETokenizer

tok = BPETokenizer.train(CORPUS)
assert tok.merges == merges and tok.vocab == vocab  # same result as the code above
print(tok.encode("I want milky ."), "→", repr(tok.decode(tok.encode("I want milky ."))))

## Your turn

Do each one **by hand first** (paper is fine), using the vocabulary and the merge list above. Then run the cell.

### 1.1 Encode "I want bed ." into IDs.

In [ ]:
my_answer = None  # a list of IDs, e.g. [1, 2, 3]
check("1.1", my_answer)

### 1.2 How does BPE cut the new word "wand"? Write the final pieces.

In [ ]:
my_answer = None  # a list of pieces, e.g. ["_ab", "c"]
check("1.2", my_answer)

In [ ]:
# Run this after answering, to see every step
encode_word("_wand", merges, verbose=True, show_misses=True);

### 1.3 Predict the pieces of the new word "tamed". Careful: is there a merge for a plain `a m`?

In [ ]:
my_answer = None  # a list of pieces
check("1.3", my_answer)

In [ ]:
# Run this after answering
encode_word("_tamed", merges, verbose=True);

### 1.4 Decode the IDs `[21, 24, 44, 22]` by hand.

In [ ]:
my_answer = None  # a string
check("1.4", my_answer)

### 1.5 Experiment: a smaller vocabulary

Real tokenizers stop at a chosen vocabulary size. Here we stop after 10 merges. With this smaller tokenizer, how many tokens is "I am hungry ."?

In [ ]:
merges_10, vocab_10 = train_bpe(CORPUS, max_merges=10)
print(len(vocab_10), "tokens; merges:", [a + b for a, b in merges_10])

my_answer = None  # a number
check("1.5", my_answer)

In [ ]:
# Run this after answering
print([piece for word in to_words("I am hungry .") for piece in encode_word(word, merges_10)])

### Optional: a real tokenizer

`tiktoken` is the tokenizer library used by OpenAI models. Install it with `.venv/bin/pip install tiktoken` (the first use also downloads its vocabulary from the internet), then compare. Notice the space glued to the front of words, like our `_`.

In [ ]:
try:
    import tiktoken
    encoding = tiktoken.get_encoding("cl100k_base")
    for text in ["I am hungry. I want milk.", "I wanted milky", "strawberry"]:
        print(f"{text!r:<28} → {[encoding.decode([i]) for i in encoding.encode(text)]}")
except ImportError:
    print("tiktoken is not installed: skip this, or run  .venv/bin/pip install tiktoken")
except Exception as error:
    print("could not load the tokenizer (it needs internet the first time):", error)

## Summary

- BPE starts from single symbols and glues the most frequent pair, again and again.
- Training gives a **vocabulary** (token → ID) and a **merge list** (the recipe).
- Encoding applies the merges in order. Decoding glues the pieces back and turns `_` into spaces.
- Common words become one token; rare or new words are cut into known pieces.
- An ID is only a row number: it carries no meaning. Giving it meaning is the job of **Step 2: embeddings** (`02_embeddings.ipynb`).